In [ ]:
# # data preparation --- data for 2021-2023, and merge with heritage data (volume)
# import pandas as pd

# # read the CSV file
# df = pd.read_csv('data_optim_add 1.csv')  #

# # 1️⃣ 找到包含 2021-2023 的 csemCur
# valid_csemCur = (
#     df[df['year'].isin([2021, 2022, 2023])]
#     .groupby('csemCur')['year']
#     .nunique()
# )

# valid_csemCur = valid_csemCur[valid_csemCur == 3].index

# # 2️⃣ 只保留这些 csemCur，并且只保留 2021-2023
# new_df = (
#     df[
#         (df['csemCur'].isin(valid_csemCur)) &
#         (df['year'].isin([2021, 2022, 2023]))
#     ]
#     .sort_values(['csemCur', 'year'])
#     .reset_index(drop=True)
# )

# # load heritage data
# donnees_patrimoniales = pd.read_csv('Donnees patrimoniales - volume.csv')

# donnees_patrimoniales['csemCur'] = pd.to_numeric(donnees_patrimoniales['csemCur'], errors='coerce')

# # merge volume column based on csemCur (left join to keep all rows in new_df)
# new_df = new_df.merge(
#     donnees_patrimoniales[['csemCur', 'Volume']], 
#     on='csemCur', 
#     how='left'
# )

# # convert csemCur back to Int
# new_df['csemCur'] = pd.to_numeric(new_df['csemCur'], errors='coerce').astype('Int64')  # use Int64 to allow for NaN values

# # save to a new CSV
# new_df.to_csv('new_df_with_volume_New.csv', index=False)

# # results
# print("New DataFrame (new_df) created successfully!")
# print(f"Shape: {new_df.shape}")
# print(f"Unique csemCur count: {new_df['csemCur'].nunique()}")
# print(new_df.head())



New DataFrame (new_df) created successfully!
Shape: (1035, 13)
Unique csemCur count: 345
      csemCur  year  delta_pressure   cr  \
0  5000195187  2021             NaN  0.0   
1  5000195187  2022             NaN  0.0   
2  5000195187  2023             NaN  0.0   
3  5000200795  2021             NaN  0.0   
4  5000200795  2022             NaN  0.0   

   compensatedRelativePressure_mafiltered  COPER  MANX     alpha  \
0                                     NaN      0     0  0.000000   
1                                3.449960      0     0  0.026605   
2                                     NaN      0     1  0.000000   
3                                     NaN      0     0  0.000000   
4                                3.623786      0     0  0.000647   

   algo_critere_exploit_B_mafiltered  algo_critere_env_B_mafiltered  \
0                              False                          False   
1                              False                          False   
2                       

In [2]:
import pandas as pd
import numpy as np
pd.set_option('display.float_format', '{:.6f}'.format)

# =============================================================================
# 0. read data with CR_threshold column added
# =============================================================================
new_df = pd.read_csv('new_df_with_volume.csv')
# # add CR_threshold column based on Pression Nominale Relative
# new_df.loc[:, 'CR_threshold'] = new_df['Pression Nominale Relative'] * 0.914

# =============================================================================
# 1. parameters
# =============================================================================
T = 17
k = 0.1
mu = 4.2 # improvement factor of MANT
base_year = 2024
        
C_MANT  = 48000      # 触发后下一年的 action cost
V = new_df['Volume'].head(T).tolist() 
K = [10**2 * 146 * V[t] / (8.314 * 293.15) for t in range(T)]    # K[i] calculation
c_factors = [0.0952 + 0.004 * t for t in range(T)]  # [0.0952, 0.0992, ..., 0.1312]
C_leak_t = [c * 25200 for c in c_factors]  # [2399.04, 2499.84, ..., 3306.24]

# =============================================================================
# 2. alpha extraction for 2023 (baseline)
# =============================================================================
new_df['csemCur'] = new_df['csemCur'].astype(int)
df_alpha_2023 = new_df.groupby("csemCur")[["alpha", "Pression Nominale Relative", "CR_threshold"]].last().reset_index()
df_alpha_2023.columns = ["csemCur", "alpha_2023", "Pression Nominale Relative", "CR_threshold"]
print("2023 Alpha Values：")
print(df_alpha_2023)
print(f"Count: {len(df_alpha_2023)} csemCur\n")

# =============================================================================
# 3. simulation with costs
# =============================================================================
rows = []
for _, row in df_alpha_2023.iterrows():
    csemCur = int(row["csemCur"])
    alpha_2023 = row["alpha_2023"]
    pression_nominale = row["Pression Nominale Relative"]
    cr_threshold = row["CR_threshold"]
    
    alpha_current = alpha_2023 * 1.01
    triggered_previous_year = False  # initially no trigger in 2023, so no action cost in 2024
    
    for t in range(T):
        year = base_year + t
        
        # calculate leak and interval
        leak = (alpha_current * 365 / (pression_nominale - cr_threshold)) * (pression_nominale - cr_threshold) * K[t]
        interval = 365 / (alpha_current * 365 / (pression_nominale - cr_threshold)) if alpha_current > 0 else np.inf
        
        # check triggers
        trigger_this_year = (leak >= 5) or (interval <= 60)
        
        # calculate costs
        leak_cost = leak * C_leak_t[t]  # leak cost varies with time
        action_cost = C_MANT if triggered_previous_year else 0
        total_cost = leak_cost + action_cost
        
        rows.append({
            "csemCur": csemCur,
            "year": year,
            "alpha": alpha_current,
            "leak": leak,
            "interval": interval,
            "triggered": trigger_this_year,
            "leak_cost": leak_cost,
            "action_cost": action_cost,
            "total_cost": total_cost,
        })
        
        # update alpha for next year
        # 🔥if trigger，next_year_alpha = current_alpha / mu（no 1% increase for this year）
        if trigger_this_year:
            alpha_next = alpha_current / mu  
        else:
            alpha_next = alpha_current * 1.01  # 1% increase if no trigger
        alpha_current = alpha_next
        triggered_previous_year = trigger_this_year

# =============================================================================
# 4. results DataFrame
# =============================================================================
sim_df = pd.DataFrame(rows)
sim_df['csemCur'] = sim_df['csemCur'].astype(int)


# =============================================================================
# 5. summary statistics
# =============================================================================
print("\n=== Summary Statistics ===")

# calculate total cost and total leak per csemCur
cost_summary = sim_df.groupby("csemCur")["total_cost"].sum().round(0).reset_index()
cost_summary.columns = ["csemCur", "total_T years_cost"]

leak_summary = sim_df.groupby("csemCur")["leak"].sum().round(3).reset_index()
leak_summary.columns = ["csemCur", "total_leak_kg"]

# calculate trigger counts for each criterion
sim_df['criterion_environment'] = (sim_df['leak'] >= 5)  # leak trigger
sim_df['criterion_exploit'] = (sim_df['interval'] <= 60)  # operational trigger

trigger_exploit = sim_df.groupby("csemCur")["criterion_exploit"].sum().astype(int).reset_index()
trigger_exploit.columns = ["csemCur", "trigger_exploit_times"]

trigger_env = sim_df.groupby("csemCur")["criterion_environment"].sum().astype(int).reset_index()
trigger_env.columns = ["csemCur", "trigger_env_times"]

# calculate total years simulated per csemCur
total_years_df = sim_df.groupby("csemCur").size().reset_index(name='total_years')

# merge all summaries into one DataFrame
summary_df = (cost_summary.merge(leak_summary, on='csemCur', how='outer')
                       .merge(trigger_exploit, on='csemCur', how='outer')
                       .merge(trigger_env, on='csemCur', how='outer')
                       .merge(total_years_df, on='csemCur', how='outer'))

# calculate trigger rates
summary_df['trigger_exploit_rate'] = (summary_df['trigger_exploit_times'] / summary_df['total_years']).round(3)
summary_df['trigger_env_rate'] = (summary_df['trigger_env_times'] / summary_df['total_years']).round(3)
summary_df['total_trigger_rate'] = ((summary_df['trigger_exploit_times'] + summary_df['trigger_env_times']) / summary_df['total_years']).round(3)

print("📊 Overall Summary:")
print(summary_df.round(3))

# print("\n📊 按泄漏量criterion触发次数排序：")
# print(summary_df.sort_values('trigger_exploit_times', ascending=False).round(3).head(10))

# print("\n📊 按环境criterion触发次数排序：")
# print(summary_df.sort_values('trigger_env_times', ascending=False).round(3).head(10))

print("\n📊 Rank by Total Leak (kg):")
print(summary_df.sort_values('total_leak_kg', ascending=False).round(3).head(10))

# Save summary statistics to CSV
# summary_df.to_csv("csemCur_summary_stats.csv", index=False)
# print("\n✅ Summary statistics saved to: csemCur_summary_stats.csv")


# =============================================================================
# 6. save detailed simulation results to CSV
# =============================================================================
sim_df.to_csv("simulation_with_costs.csv", index=False)
print("\n✅ Simulation completed!")
print("✅ Results saved to: simulation_with_costs.csv")
print("✅ Preview of first 10 rows:")
print(sim_df.head(10).round({'alpha':6, 'leak':3, 'leak_cost':0, 'total_cost':0}))


2023 Alpha Values：
        csemCur  alpha_2023  Pression Nominale Relative  CR_threshold
0    5000200795    0.004915                    3.500000      3.199000
1    5000252448    0.181613                    3.500000      3.199000
2    5000252449    0.022172                    3.500000      3.199000
3    5000252450    0.018647                    3.500000      3.199000
4    5000252524    0.032852                    3.500000      3.199000
..          ...         ...                         ...           ...
590  5002343706    0.121390                    3.500000      3.199000
591  5002343707    0.063426                    3.500000      3.199000
592  5002699073    0.019229                    3.500000      3.199000
593  5002903325    0.001703                    3.500000      3.199000
594  5002903326    0.030425                    3.500000      3.199000

[595 rows x 4 columns]
Count: 595 csemCur


=== Summary Statistics ===
📊 Overall Summary:
        csemCur  total_T years_cost  total_leak_kg

In [6]:
import pandas as pd
import numpy as np
pd.set_option('display.float_format', '{:.6f}'.format)


# =============================================================================
# 0. read data with CR_threshold column added
# =============================================================================
new_df = pd.read_csv('new_df_with_volume.csv')
# # add CR_threshold column based on Pression Nominale Relative
# new_df.loc[:, 'CR_threshold'] = new_df['Pression Nominale Relative'] * 0.914


# =============================================================================
# 1. parameters
# =============================================================================
T = 17
k = 0.1
mu = 4.2 # improvement factor of MANT
base_year = 2024
        
C_MANT  = 48000      # 触发后下一年的 action cost
c_factors = [0.0952 + 0.004 * t for t in range(T)]  # [0.0952, 0.0992, ..., 0.1312]
C_leak_t = [c * 25200 for c in c_factors]  # [2399.04, 2499.84, ..., 3306.24]


# =============================================================================
# 2. alpha extraction for 2023 (baseline) + Volume
# =============================================================================
new_df['csemCur'] = new_df['csemCur'].astype(int)
df_alpha_2023 = new_df.groupby("csemCur")[["alpha", "Pression Nominale Relative", "CR_threshold", "Volume"]].last().reset_index()
df_alpha_2023.columns = ["csemCur", "alpha_2023", "Pression Nominale Relative", "CR_threshold", "Volume"]
print("2023 Alpha Values：")
print(df_alpha_2023)
print(f"Count: {len(df_alpha_2023)} csemCur\n")


# =============================================================================
# 3. simulation with costs
# =============================================================================
rows = []
for _, row in df_alpha_2023.iterrows():
    csemCur = int(row["csemCur"])
    alpha_2023 = row["alpha_2023"]
    pression_nominale = row["Pression Nominale Relative"]
    cr_threshold = row["CR_threshold"]
    volume_csem = row["Volume"]  # 🔥 每个csemCur固定Volume
    
    alpha_current = alpha_2023 * 1.01
    triggered_previous_year = False
    
    # 🔥 csemCur固定K（多年相同）
    csem_K = 10**2 * 146 * volume_csem / (8.314 * 293.15)
    
    for t in range(T):
        year = base_year + t
        
        # calculate leak and interval（固定K，多年不变）
        leak = alpha_current * 365 * csem_K
        delta_p = pression_nominale - cr_threshold
        interval = 365 / (alpha_current * 365 / delta_p) if alpha_current > 0 else np.inf
        
        # check triggers
        trigger_this_year = (leak >= 5) or (interval <= 60)
        
        # calculate costs
        leak_cost = leak * C_leak_t[t]  # leak cost varies with time
        action_cost = C_MANT if triggered_previous_year else 0
        total_cost = leak_cost + action_cost
        
        rows.append({
            "csemCur": csemCur,
            "year": year,
            "alpha": alpha_current,
            "leak": leak,
            "interval": interval,
            "triggered": trigger_this_year,
            "leak_cost": leak_cost,
            "action_cost": action_cost,
            "total_cost": total_cost,
        })
        
        # update alpha for next year
        if trigger_this_year:
            alpha_next = alpha_current / mu  
        else:
            alpha_next = alpha_current * 1.01
        alpha_current = alpha_next
        triggered_previous_year = trigger_this_year


# =============================================================================
# 4-6. 后续部分完全不变
# =============================================================================
sim_df = pd.DataFrame(rows)
sim_df['csemCur'] = sim_df['csemCur'].astype(int)

print("\n=== Summary Statistics ===")

cost_summary = sim_df.groupby("csemCur")["total_cost"].sum().round(0).reset_index()
cost_summary.columns = ["csemCur", "total_T years_cost"]

leak_summary = sim_df.groupby("csemCur")["leak"].sum().round(3).reset_index()
leak_summary.columns = ["csemCur", "total_leak_kg"]

sim_df['criterion_environment'] = (sim_df['leak'] >= 5)
sim_df['criterion_exploit'] = (sim_df['interval'] <= 60)

trigger_exploit = sim_df.groupby("csemCur")["criterion_exploit"].sum().astype(int).reset_index()
trigger_exploit.columns = ["csemCur", "trigger_exploit_times"]

trigger_env = sim_df.groupby("csemCur")["criterion_environment"].sum().astype(int).reset_index()
trigger_env.columns = ["csemCur", "trigger_env_times"]

total_years_df = sim_df.groupby("csemCur").size().reset_index(name='total_years')

summary_df = (cost_summary.merge(leak_summary, on='csemCur', how='outer')
                       .merge(trigger_exploit, on='csemCur', how='outer')
                       .merge(trigger_env, on='csemCur', how='outer')
                       .merge(total_years_df, on='csemCur', how='outer'))

summary_df['trigger_exploit_rate'] = (summary_df['trigger_exploit_times'] / summary_df['total_years']).round(3)
summary_df['trigger_env_rate'] = (summary_df['trigger_env_times'] / summary_df['total_years']).round(3)
summary_df['total_trigger_rate'] = ((summary_df['trigger_exploit_times'] + summary_df['trigger_env_times']) / summary_df['total_years']).round(3)

print("📊 Overall Summary:")
print(summary_df.round(3))

print("\n📊 Rank by Total Leak (kg):")
print(summary_df.sort_values('total_leak_kg', ascending=False).round(3).head(10))

sim_df.to_csv("simulation_with_costs.csv", index=False)
print("\n✅ Simulation completed!")
print("✅ Results saved to: simulation_with_costs.csv")
print("✅ Preview of first 10 rows:")
print(sim_df.head(18).round({'alpha':6, 'leak':3, 'leak_cost':0, 'total_cost':0}))


2023 Alpha Values：
        csemCur  alpha_2023  Pression Nominale Relative  CR_threshold  \
0    5000200795    0.004915                    3.500000      3.199000   
1    5000252448    0.181613                    3.500000      3.199000   
2    5000252449    0.022172                    3.500000      3.199000   
3    5000252450    0.018647                    3.500000      3.199000   
4    5000252524    0.032852                    3.500000      3.199000   
..          ...         ...                         ...           ...   
590  5002343706    0.121390                    3.500000      3.199000   
591  5002343707    0.063426                    3.500000      3.199000   
592  5002699073    0.019229                    3.500000      3.199000   
593  5002903325    0.001703                    3.500000      3.199000   
594  5002903326    0.030425                    3.500000      3.199000   

       Volume  
0    7.583613  
1   63.196772  
2   63.053685  
3   98.110008  
4    4.435698  
..      